# E09_CurriculumMaze — PPO с учебным планом

**Урок:** 3.3 «Учебный план и рандомизация среды» курса
[cyber-unity-learn](https://rl-cuber-unity-code.com/courses/3-3)
**Что реализуется:** PPO с категориальной политикой плюс учебный план,
который повышает сложность лабиринта по мере роста доли успехов.
**Ожидаемый результат:** **≥ 60 %** пройденных лабиринтов на **фиксированной
максимальной** сложности (не на текущем уроке).
**Время обучения:** около 35 минут на сид (`QUICK_RUN = True` — около трёх минут).

## Что здесь принципиально нового

**Задача меняется по ходу обучения.** Во всех предыдущих средах лаборатории
MDP был неизменен. Здесь размер лабиринта и плотность стен зависят от числа
`difficulty`, которое Python передаёт в Unity через
`EnvironmentParametersChannel`, а среда читает на границе каждого эпизода.

**Формирования награды здесь нет — намеренно.** В `E06_Hunter3D`
и `E08_SoccerArena` плотный сигнал давал потенциал по расстоянию до цели.
В лабиринте такой потенциал **вреден**: он тянет агента к стене, за которой
выход, а обходной путь ведёт «от» выхода и штрафуется. Безопасного плотного
сигнала для этой задачи нет — и именно поэтому нужен учебный план.

**Каждый эпизод — новая карта.** Это Domain Randomization: при неизменной
планировке агент выучил бы **её**, а не умение проходить лабиринты, и
на новой карте рассыпался бы.

## Главное следствие для чтения графиков

**Доля успехов на текущем уроке почти ничего не говорит.** Она по построению
держится около порога повышения: как только агент начинает справляться,
план даёт задачу труднее, и метрика падает обратно. График `Eval/Success Rate`
здесь пилообразный, и это норма.

Мера прогресса — **оценка на фиксированной максимальной сложности**
(раздел 8). Она же критерий приёмки. Без этого разделения прогон, застрявший
на первом уроке, и прогон, дошедший до последнего, показывают одинаковые 70 %.

## Требования

```powershell
python scripts\build_env.py E09_CurriculumMaze
```

## 1. Окружение

In [ ]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

## 2. Конфигурация

Значения совпадают с `configs/E09_CurriculumMaze__ppo_curriculum.yaml`.

Учебный план — четыре урока. Три решения в нём легко принять неправильно:

| Решение | Почему так |
|---|---|
| измеряется **доля успехов**, а не награда | награда зависит от сложности: на большой сетке путь длиннее и штраф за время больше, и один порог означал бы разные вещи на разных уроках |
| пороги **убывают** с ростом сложности | требовать 70 % на сетке 11 × 11 — значит не дойти до неё никогда |
| план **не откатывается** назад | доля успехов падает после каждого повышения; откат по такому сигналу даёт колебания между двумя уровнями |

`min_steps` защищает от перевода по одному удачному замеру: доля успехов
по трём десяткам эпизодов заметно шумит.

In [ ]:
QUICK_RUN = True   # True — быстрая проверка (~3 мин); False — полное обучение (~35 мин)

from labrl.algos.ppo import PPOConfig
from labrl.eval.success import TERMINATED
from labrl.train.curriculum import Curriculum, Lesson
from labrl.train.onpolicy import OnPolicyTrainConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 30_000 if QUICK_RUN else 300_000

ALGO_CFG = PPOConfig(
    gamma=0.99,
    gae_lambda=0.95,
    learning_rate=3e-4,
    clip_range=0.2,
    epochs=4,
    minibatch_size=512,
    value_coef=0.5,
    entropy_coef=0.01,       # выше, чем у сред с плотной наградой: выход надо найти
    max_grad_norm=0.5,
    normalize_advantage=True,
    target_kl=0.0,
)

TRAIN_CFG = OnPolicyTrainConfig(
    total_steps=TOTAL_STEPS,
    rollout_steps=256,                          # 256 шагов × 8 арен = 2048 переходов
    eval_every_steps=max(1, TOTAL_STEPS // 10),
    eval_episodes=30,
    success_rule=TERMINATED,                    # успех = выход найден, а не «награда выше порога»
    success_threshold=0.0,
)

# Уроки. В быстром прогоне min_steps сжимаются пропорционально — иначе план
# не сдвинется с первого урока и проверять будет нечего. Это единственное
# место, где сокращённый бюджет меняет расписание, и причина у него другая,
# чем в T-10: там сжатие ломало метод, здесь без сжатия смоук-тест
# не проверял бы то, ради чего он есть.
SCALE = 10 if QUICK_RUN else 1
CURRICULUM = Curriculum([
    Lesson(difficulty=0.00, threshold=0.85, min_steps=20_000 // SCALE),   # сетка 5×5, случайно 70 %
    Lesson(difficulty=0.33, threshold=0.65, min_steps=30_000 // SCALE),   # сетка 7×7, случайно 32 %
    Lesson(difficulty=0.67, threshold=0.60, min_steps=40_000 // SCALE),   # сетка 9×9, случайно 7 %
    Lesson(difficulty=1.00, threshold=1.00),                              # сетка 11×11, случайно 5 %
])

# Сложность приёмочной оценки. Фиксирована и не зависит от того, до какого
# урока дошёл план: сравнивать сиды между собой можно только на одной задаче.
EVAL_DIFFICULTY = 1.0

# Расписание шага обучения НЕ сжимается вместе с бюджетом (T-10).
LEARNING_RATE = LinearSchedule(start=3e-4, end=5e-5, decay_steps=300_000)

print(f"бюджет: {TOTAL_STEPS} шагов, обновлений: {TOTAL_STEPS // TRAIN_CFG.rollout_steps}")
print(f"уроков: {len(CURRICULUM.lessons)}, приёмка на сложности {EVAL_DIFFICULTY:.2f}")

## 3. Подключение среды

Стартовая сложность — сложность первого урока. Дальше её меняет план,
а не конфиг среды.

In [ ]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E09_CurriculumMaze"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(
    ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True,
    num_areas=8, env_parameters={"difficulty": CURRICULUM.difficulty},
)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")
print(f"стартовая сложность: {CURRICULUM.difficulty:.2f}")

## 4. Инспекция пространств

Проверка против строки-контракта `ENV_SPEC.md`. Обратите внимание на имя
первого сенсора: `StackingSensor_size2_RayPerceptionSensor` — два кадра лучей
склеены самим ML-Agents, и политика видит 40 чисел, а не 20.

In [ ]:
OBS_DIM = vec.single_obs_dim          # склейка сенсоров: 40 + 10
BRANCHES = tuple(vec.action_spec.discrete_branches)

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"имена сенсоров:    {handle.obs_names}")
print(f"obs_dim (склейка): {OBS_DIM}")
print(f"дискретные ветки:  {BRANCHES}")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=10; discrete_branches=4; continuous_size=0; max_step=150 -->
assert [tuple(s) for s in vec.obs_shapes] == [(40,), (10,)], (
    f"порядок или размер сенсоров не тот: {vec.obs_shapes}")
assert OBS_DIM == 50, f"obs_dim {OBS_DIM} != 50"
assert BRANCHES == (4,), f"ветки {BRANCHES} != (4,)"
assert vec.action_spec.continuous_size == 0

print(f"\nobs_1 (вектор, 10): {np.array2string(obs[1][0], precision=2)}")
print("  [0:2] клетка агента, [2:4] клетка выхода, [4:6] смещение,")
print("  [6] манхэттен, [7] доля времени, [8] difficulty, [9] размер сетки")
print("\nразмерности и порядок сенсоров совпадают с ENV_SPEC.md")

## 5. Нейросеть

Категориальная политика: сеть отдаёт четыре логита, по одному на направление.
Сэмплирование живёт в алгоритме и в графе ONNX, а не в сети — иначе одну
и ту же логику пришлось бы поддерживать в двух местах.

In [ ]:
from labrl.nets.categorical_policy import MultiBranchCategoricalPolicy


class MyPolicy(MultiBranchCategoricalPolicy):
    """Актор: obs (B, 50) -> логиты (B, 4)."""

    def __init__(self, obs_dim: int, branches: tuple[int, ...], hidden: int = 256):
        super().__init__(obs_dim, branches, (hidden, hidden), "relu")


class MyCritic(nn.Module):
    """Критик: obs (B, 50) -> V(s) (B, 1). В Unity не уезжает."""

    def __init__(self, obs_dim: int, hidden: int = 256):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs):
        return self.body(obs)


policy_net = MyPolicy(OBS_DIM, BRANCHES)
value_net = MyCritic(OBS_DIM)

with torch.no_grad():
    probe = torch.zeros(3, OBS_DIM)
    assert policy_net(probe).shape == (3, sum(BRANCHES))
    assert value_net(probe).shape == (3, 1)

print(policy_net)
print(f"\nпараметров актора:  {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"параметров критика: {sum(p.numel() for p in value_net.parameters()):,}")
print(f"стартовая энтропия: {float(policy_net.entropy(probe)):.3f} нат "
      f"(максимум ln4 = {np.log(4):.3f})")

## 6. Теория: почему учебный план, а не формирование награды

Потенциальное формирование награды (Ng, Harada, Russell, 1999) даёт плотный
сигнал, не меняя множества оптимальных политик:

$$F(s, s') = \gamma\,\Phi(s') - \Phi(s)$$

Теорема верна для **любого** $\Phi$ — но полезность зависит от того,
насколько $\Phi$ похож на настоящую функцию ценности. В лабиринте очевидный
кандидат $\Phi(s) = -d_{\text{евкл}}(s)$ похож на неё плохо:

```
   агент  →  ███████████  ←  за стеной выход
             ███████████
             ┌─────────┐
             обходной путь: d_евкл РАСТЁТ, F < 0
```

Обход стены увеличивает евклидово расстояние, и формирование штрафует
единственный правильный путь. Это не ошибка теоремы — оптимальная политика
действительно не меняется, — но плотный сигнал ведёт **в противоположную
сторону**, и обучение от него становится медленнее, а не быстрее.

Правильный потенциал — длина пути из поиска в ширину — среде известна
(`MazeArea.ShortestPath`), но агенту недоступна: если бы он её знал,
задачи бы не было.

Учебный план не трогает награду. Он делает так, чтобы **редкая награда
всё-таки случалась**: на сетке 5 × 5 случайная политика доходит до выхода
регулярно, и сигнал появляется с первого эпизода.

Ниже — правило перевода, тот самый код, что выполняется при обучении.

In [ ]:
import inspect

from labrl.train.curriculum import Curriculum

print(inspect.getsource(Curriculum.report))

## 7. Цикл обучения

Учебный план подключается через колбэк `on_eval`: цикл обучения о нём
ничего не знает, а план получает свежий замер доли успехов и, если урок
сменился, кладёт новую сложность в боковой канал.

In [ ]:
from labrl.algos.ppo_discrete import PPODiscrete
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.onpolicy import train_on_policy

algo = PPODiscrete(policy_net, value_net, ALGO_CFG, device=DEVICE, seed=SEED)
run = create_run_dir(ENV_ID, "ppo_discrete", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")


def on_eval(step: int, reward: float, success: float) -> None:
    print(f"  шаг {step:>7}: награда {reward:>7.3f}, пройдено {success:.0%} "
          f"(урок {CURRICULUM.index + 1}, difficulty {CURRICULUM.difficulty:.2f})")
    if CURRICULUM.report(step, success):
        # Урок сменился — новую сложность нужно донести до среды.
        handle.channels.set_difficulty(CURRICULUM.difficulty)
        print(f"    ── переход на урок {CURRICULUM.index + 1}: "
              f"difficulty = {CURRICULUM.difficulty:.2f}")


result = train_on_policy(
    vec=vec, algo=algo, logger=logger, cfg=TRAIN_CFG, seed=SEED,
    lr_schedule=LEARNING_RATE, on_eval=on_eval,
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={
    "seed": SEED, "curriculum": CURRICULUM.history,
    "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, обновлений: {result.updates}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")
print()
print(CURRICULUM.describe())

## 8. Оценка на фиксированной сложности

Здесь и только здесь измеряется то, что можно сравнивать между сидами
и прогонами. Оценка идёт **той же** средой: сложность выставляется в боковой
канал, среда подхватывает её на границе следующего эпизода, а прогретые
эпизоды отбрасываются.

In [ ]:
from labrl.train.curriculum import evaluate_at_difficulty

outcome = evaluate_at_difficulty(
    vec, algo, handle.channels, vec.reset(), EVAL_DIFFICULTY,
    episodes=TRAIN_CFG.eval_episodes,
    success_rule=TRAIN_CFG.success_rule,
)

print(f"сложность оценки:   {EVAL_DIFFICULTY:.2f} (учебный план дошёл до "
      f"{CURRICULUM.difficulty:.2f})")
print(f"пройдено лабиринтов: {outcome.success_rate:.0%}")
print(f"средняя награда:     {outcome.mean_reward:+.3f}")
print(f"средняя длина:       {outcome.mean_length:.1f} ходов")

# Критерий приёмки проверяется только на ПОЛНОМ бюджете: сокращённый прогон —
# смоук-тест конвейера, на нём метод не обязан обучиться.
THRESHOLD = 0.60   # блок success_criteria конфига, метрика Eval/Success Rate

if QUICK_RUN:
    print(f"\nбыстрый прогон: критерий приёмки ({THRESHOLD:.0%}) не проверяется — "
          "проверялась работоспособность конвейера")
else:
    assert outcome.success_rate >= THRESHOLD, (
        f"не достигнут критерий приёмки: {outcome.success_rate:.0%} < {THRESHOLD:.0%}")
    print(f"\nкритерий приёмки достигнут: {outcome.success_rate:.0%} >= {THRESHOLD:.0%}")

### Кривые обучения

In [ ]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

steps = [s for s, _, _ in result.eval_history]
success = [v for _, _, v in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]

axes[0].plot(steps, success, marker="o", color="tab:green")
for change_step, index, _ in CURRICULUM.history:
    axes[0].axvline(change_step, color="tab:red", ls="--", lw=1)
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("Доля успехов на ТЕКУЩЕМ уроке\n(пунктир — переходы плана)")
axes[0].set_xlabel("шаг сбора"); axes[0].grid(alpha=0.3)

# Ступеньки учебного плана: то же, что Env/Difficulty в TensorBoard.
plan_steps, plan_difficulty = [0], [CURRICULUM.lessons[0].difficulty]
for change_step, index, _ in CURRICULUM.history:
    plan_steps += [change_step, change_step]
    plan_difficulty += [plan_difficulty[-1], CURRICULUM.lessons[index].difficulty]
plan_steps.append(TOTAL_STEPS); plan_difficulty.append(plan_difficulty[-1])

axes[1].plot(plan_steps, plan_difficulty, color="tab:blue", drawstyle="default")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("Учебный план: difficulty")
axes[1].set_xlabel("шаг сбора"); axes[1].grid(alpha=0.3)

axes[2].plot(steps, rewards, marker="o", color="tab:orange")
axes[2].set_title("Награда на текущем уроке\n(зависит от сложности — не сравнима)")
axes[2].set_xlabel("шаг сбора"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Экспорт в ONNX и верификация

Категориальная политика с одной веткой из четырёх действий: `discrete_actions`
получается сэмплированием из softmax, `deterministic_discrete_actions` —
argmax. Оба выхода **целые**: Unity читает их как `Tensor<int>`, и приведение
к float ломает инференс, оставляя ONNX формально валидным.

In [ ]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из реального распределения среды (требование 10.5), а не шум.
collected = [[] for _ in vec.obs_shapes]
obs_now = vec.reset()
while sum(len(c) for c in collected[0]) < 64:
    for sensor, batch in enumerate(obs_now):
        collected[sensor].append(batch.copy())
    obs_now = vec.step(algo.deterministic_action(vec.flatten_obs(obs_now))).obs
sample = [np.concatenate(parts, axis=0)[:64] for parts in collected]

spec = ActionSpecLite(discrete_branches=BRANCHES, continuous_size=0)
policy = algo.policy_module() if hasattr(algo, "policy_module") else policy_net.eval()
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx",
                                  strategy="categorical")

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

# Python и граф обязаны выбирать одно и то же действие.
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
graph_action = session.run(
    ["deterministic_discrete_actions"],
    {"obs_0": sample[0], "obs_1": sample[1],
     "action_masks": np.ones((64, sum(BRANCHES)), dtype=np.float32),
     "recurrent_in": np.zeros((64, 1, 0), dtype=np.float32)},
)[0]
python_action = algo.deterministic_action(np.concatenate(sample, axis=1))
mismatch = int((graph_action != python_action).sum())
print(f"\nрасхождений действия ONNX и Python: {mismatch} из {python_action.size}")
assert mismatch == 0, "оценка в Python и граф расходятся — инференс в Unity будет другим"

if not QUICK_RUN:
    # Модель проекта обновляется только полным прогоном: быстрый затёр бы её
    # смоук-версией (T-14 в docs/07_TROUBLESHOOTING.md).
    model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
                  "Models" / f"{ENV_ID}_ppo.onnx")
    model_dest.parent.mkdir(parents=True, exist_ok=True)
    model_dest.write_bytes(onnx_path.read_bytes())
    print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")
else:
    print("быстрый прогон: модель в проект Unity НЕ копируется (T-14)")

vec.handle.close()

## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E09_CurriculumMaze__ppo_curriculum.yaml --python-reward <оценка из раздела 8>
```

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E09_CurriculumMaze/Scenes/E09_CurriculumMaze.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента `Agent`: `Model` = импортированная модель,
   `Behavior Type` = `Inference Only`, галочка `Deterministic Inference`.
5. Play: **лабиринт будет 11 × 11** — без подключённого Python параметр
   `difficulty` передать некому, и среда берёт `defaultDifficulty` компонента
   `MazeArea`. Setup-скрипт его не задаёт, поэтому действует значение
   по умолчанию `TrainingAreaBase` — 1.0. Чтобы посмотреть лёгкий уровень,
   задайте `defaultDifficulty = 0` в инспекторе.

### Если что-то идёт не так

| Симптом | Где искать |
|---|---|
| агент упирается в стену и стоит | штраф за стену слишком мал либо лучи не видят стен — проверить `DetectableTags` |
| в редакторе всегда сетка 11 × 11 | это норма: без Python `difficulty` берётся из `defaultDifficulty` = 1.0 |
| доля успехов упирается в потолок ниже 1.0 | часть лабиринтов нерешаема — проверить BFS в `MazeArea.GenerateSolvableMaze` |

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, стрелки или **WASD** — одно нажатие,
один ход.